# Development evaluation on Colab

Select a **T4 GPU** runtime. This notebook reuses run 05 and the **12 human-reviewed development extracts**. Minerva scores their operational passages; Qwen reviews all 12 once so routing policies can be compared offline without repeated model generation. This is development calibration, not a held-out test or a full-corpus LLM run.

Minerva access and the Colab Secret `HF_TOKEN` must already be enabled. Never put the token in a cell or commit it.


In [ ]:
import os
from pathlib import Path
import subprocess
import sys

repo = Path("/content/forensic-llm-graph")
if repo.exists():
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/AntoFratta/forensic-llm-graph.git", str(repo)], check=True)
os.chdir(repo)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
print("Python:", sys.version.split()[0])


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)


In [ ]:
import json
from google.colab import userdata
from huggingface_hub import hf_hub_download

# The environment passes authentication to the child process without writing credentials.
token = userdata.get("HF_TOKEN")
if not token:
    raise RuntimeError("Add the HF_TOKEN Secret and enable notebook access.")
os.environ["HF_TOKEN"] = token
del token
settings = json.loads(Path("configs/hybrid.json").read_text(encoding="utf-8"))
hf_hub_download(settings["perplexity"]["model"], "config.json",
                revision=settings["perplexity"]["revision"])
print("Minerva access verified. No model weights loaded yet.")


## Upload the two input files

Using Colab's Files panel, upload these files to `/content/`:

- `05_batched_form_boundaries_spacy_lg.json`
- `development_reference.reviewed.json`, exported locally under `research/evaluation/`

The reference contains the reviewed annotations. Inference receives only the original passage text; labels are used separately by the evaluation script. Uploads are temporary, outside Git. No Drive mount or raw-data download is needed.


In [ ]:
input_path = Path("/content/05_batched_form_boundaries_spacy_lg.json")
reference_path = Path("/content/development_reference.reviewed.json")
for path in (input_path, reference_path):
    if not path.is_file():
        raise FileNotFoundError(f"Upload {path.name} using the Files panel before continuing.")
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)
preview_path = Path("outputs/09_development_preview.json")
subprocess.run([sys.executable, "scripts/run_hybrid.py", "--input", str(input_path),
                "--reference", str(reference_path), "--development-all", "--prepare-only",
                "--output", str(preview_path), "--overwrite"], check=True)
preview = json.loads(preview_path.read_text(encoding="utf-8"))
print("Scope:", preview["run"]["scope"])
print("Input documents:", preview["run"]["input_document_count"])
print("Full-corpus passages:", preview["run"]["full_passage_count"])
print("Development passages:", preview["run"]["passages_by_source"])
assert preview["run"]["scope"] == "development_all_review"
assert preview["run"]["passages_by_source"] == {
    "redit": 3, "archivio_antimafia": 3, "dice": 3, "rizzo_synthetic": 3
}
del preview


## Run both models on development

Minerva loads first and scores the 12 development passages, then it is released before Qwen loads. Qwen reviews every development passage to support offline routing calibration. This does not mean all corpus passages will be sent to the LLM.

The first Qwen download contains approximately 16.4 GB of source weights even though GPU loading uses four-bit quantization. Model downloads are reused within an active runtime. Progress and loading messages appear below.

Completed records are saved progressively. If interrupted, add `--resume` to the same command. An existing final JSON remains protected; choose a new output if intentionally repeating a completed run.


In [ ]:
output_path = Path("outputs/09_development_models.json")
command = [sys.executable, "-u", "scripts/run_hybrid.py", "--input", str(input_path),
           "--reference", str(reference_path), "--development-all", "--output", str(output_path)]
# To continue an interrupted run in this runtime: command.append("--resume")
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        return_code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)


In [ ]:
evaluation_path = Path("outputs/09_development_evaluation.json")
calibrated_config_path = Path("outputs/09_calibrated_hybrid.json")
subprocess.run([sys.executable, "scripts/evaluate.py", "--reference", str(reference_path),
                "--baseline", str(input_path), "--hybrid", str(output_path),
                "--calibrate-routing", "--calibrated-config", str(calibrated_config_path),
                "--output", str(evaluation_path), "--overwrite"], check=True)
results = json.loads(output_path.read_text(encoding="utf-8"))
run = results["run"]
print("Scope:", run["scope"])
print("Reviewed passages:", run["reviewed_passage_count"])
print("Perplexity status:", run["perplexity_status_counts"])
print("LLM status:", run["review_status_counts"])
print("Generation attempts:", run["generation_attempt_count"])
print("GPU:", run["device"])
print("Elapsed seconds:", round(run["invocation_seconds"], 2))
print("Saved:", output_path, evaluation_path, calibrated_config_path, sep="\n")


## Save and disconnect

Download these three files manually:

- `outputs/09_development_models.json`
- `outputs/09_development_evaluation.json`
- `outputs/09_calibrated_hybrid.json`

The `.checkpoints.jsonl` file is needed for recovery only if the model run is interrupted. Save it manually before deleting a runtime when necessary. Once the three final files are saved, disconnect while reviewing results or making local changes.

Evaluation uses exact entity boundaries and categories in reviewed cores. REDIT forms, news and judicial texts are reported separately from generated synthetic diagnostics. Undefined metrics are null and category counts must accompany interpretation. Calibration compares a small fixed set of development perplexity quantiles and keeps the same review budget. The no-perplexity comparison also removes perplexity-based tie-breaking.

Routing calibration applies to this small source-balanced reference subset, not to the full corpus distribution. The two held-out review batches remain for the test after the method and grouping have been fixed. No model training or fine-tuning is performed.
